# ELENDRIL V8 — Typed Action / Fact-Pointer Screen (Kaggle 2xT4)

Execution notebook pinned to immutable runtime commit:
- `SOURCE_SHA = "b87ff909d9e91bf60f17661fbbea49c76978f53e"`
- Owner-armed defaults (`RUN_STUDY = True`) for immediate Kaggle 2xT4 execution (set `RUN_STUDY = False` for inert dry-run).


In [ ]:
# Owner-controlled switches (RUN_STUDY=True is owner-armed for Kaggle 2xT4; set False for inert dry-run)
RUN_STUDY = True
PHASE = "data-control"  # archive-diagnostic | data-control | head-screen | reader-screen | replication
MINUTES = 330
WINNER_ARM = None
COMPARATOR_ARM = None
CONFIRMATION_SEED = None
SOURCE_SHA = "b87ff909d9e91bf60f17661fbbea49c76978f53e"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"


In [ ]:
import os, shutil, subprocess, sys, tarfile
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

WORK_ROOT = Path("/kaggle/working/elendril_v8")
REPO_DIR = WORK_ROOT / "repo"
OUTPUT_DIR = Path("/kaggle/working")

if not RUN_STUDY:
    print("NOT LAUNCHED: owner switch is false")
else:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    local_tar = Path(f"/kaggle/input/elendril-v8-source/{SOURCE_SHA}.tar.gz")
    if local_tar.exists():
        tar_path = WORK_ROOT / f"{SOURCE_SHA}.tar.gz"
        shutil.copy2(local_tar, tar_path)
        with tarfile.open(tar_path, "r:gz") as tf:
            tf.extractall(REPO_DIR)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "init", "-q"], check=True)
        remotes = subprocess.run(
            ["git", "-C", str(REPO_DIR), "remote"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.split()
        if "origin" in remotes:
            subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
        else:
            subprocess.run(["git", "-C", str(REPO_DIR), "remote", "add", "origin", REPO_URL], check=True)
        try:
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", SOURCE_SHA], check=True)
        except subprocess.CalledProcessError:
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", SOURCE_SHA], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_SHA], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    prev_pp = os.environ.get("PYTHONPATH", "")
    pp_parts = [p for p in prev_pp.split(os.pathsep) if p] if prev_pp else []
    if str(REPO_DIR) not in pp_parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), *pp_parts]) if pp_parts else str(REPO_DIR)
    print("Checked out immutable runtime:", SOURCE_SHA)


In [ ]:
import torch
if RUN_STUDY:
    gpu_count = torch.cuda.device_count()
    print("Torch:", torch.__version__, "CUDA GPUs:", gpu_count)
    for idx in range(gpu_count):
        print(f"  cuda:{idx} -> {torch.cuda.get_device_name(idx)}")
    assert torch.cuda.is_available(), "Kaggle GPU accelerator must be enabled"
    if gpu_count < 2:
        print(f"WARNING: expected 2xT4 GPUs, found {gpu_count} CUDA device(s); continuing on available GPU(s)")


In [ ]:
import os, subprocess, sys
if not RUN_STUDY:
    print("NOT LAUNCHED: owner switch is false")
else:
    cmd = [
        sys.executable, "-m", "tools.elendril_v8_campaign",
        "--phase", PHASE,
        "--source", SOURCE_SHA,
        "--work-root", str(WORK_ROOT / "runs"),
        "--output", str(OUTPUT_DIR),
        "--minutes", str(MINUTES),
        "--owner-enabled",
    ]
    if WINNER_ARM:
        cmd.extend(["--winner-arm", str(WINNER_ARM)])
    if COMPARATOR_ARM:
        cmd.extend(["--comparator-arm", str(COMPARATOR_ARM)])
    if CONFIRMATION_SEED is not None:
        cmd.extend(["--confirmation-seed", str(CONFIRMATION_SEED)])
    env = dict(os.environ)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    env["OMP_NUM_THREADS"] = "2"
    env["MKL_NUM_THREADS"] = "2"
    prev_pp = env.get("PYTHONPATH", "")
    pp_parts = [p for p in prev_pp.split(os.pathsep) if p] if prev_pp else []
    if str(REPO_DIR) not in pp_parts:
        env["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), *pp_parts]) if pp_parts else str(REPO_DIR)
    subprocess.run(cmd, env=env, cwd=str(REPO_DIR), check=True)


In [ ]:
import json, sys

if not RUN_STUDY:
    print("NOT LAUNCHED: owner switch is false")
else:
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    from IPython.display import FileLink, display
    from tools.elendril_v8_export import verify_v8_export

    receipt = verify_v8_export(OUTPUT_DIR, expected_source_commit=SOURCE_SHA)
    print("Verified exports:", json.dumps(receipt, indent=2))
    for fname in ("FINAL_STATUS.json", "ELENDRIL_V8_ANALYSIS.zip", "ELENDRIL_V8_FULL.zip"):
        if (OUTPUT_DIR / fname).exists():
            display(FileLink(str(OUTPUT_DIR / fname)))
